# Creating an even centric knowledge graph

This notebook describes how an event-centric Knowledge Graph can be created by pushing capsules with event details to a GraphDB knowledge graph. Check out the README.md to get started. This notebooks assumed you installed the packages from the ```requirements.txt```in a separate conda environment and run this notebook in that environment.

## Dependencies imported

In [2]:
import json
import requests
from cltl.brain.long_term_memory import LongTermMemory
from cltl.commons.discrete import UtteranceType
from tqdm import tqdm
from random import getrandbits
from pathlib import Path
from dateutil import parser
from datetime import date, datetime
from tqdm import tqdm

## Setting up the Knowledge Graph end point

The README explain how to set up the Knowledge Graph in GraphDB. GraphDB needs to be running and a repository with the name ```event_sandbox``` need to be defined. In the notebook, we create a LongTermMemory instance, called ```brain``` that is used to push the conversational data and event triples to the repository. To create the instance, three parameters are required:

1. the address of the repository in GraphDB
2. the path to a folder (```graph```) to log the RDF triples that are pushed, which is created if not present
3. a boolean indicating if the repository needs to be emptied or not when calling

In [3]:
### Initialisation of the path for logging and of the GraDB repository for saving the data
# Create folders

graph_filepath = Path('./graph')
graph_filepath.mkdir(parents=True, exist_ok=True)

# Create brain connection
brain = LongTermMemory(address="http://localhost:7200/repositories/event_sandbox",  # Location to save accumulated graph
                       log_dir=graph_filepath,  # Location to save step-wise graphs
                       clear_all=True)  # To start from an empty brain

2026-06-02 17:08:13 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-06-02 17:08:13 -     INFO -                                    cltl.brain.LongTermMemory - Clearing brain
2026-06-02 17:08:13 -     INFO -                                    cltl.brain.LongTermMemory - Uploading ontology to brain
2026-06-02 17:08:14 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-06-02 17:08:14 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-06-02 17:08:14 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-06-02 17:08:14 -     INFO -                                   cltl.brain.TrustCalculator - Booted


When empty, the Knowledge Graph is first preloaded with a so-called T-box that is given in: [cltl-knowledgerepresentation](https://github.com/leolani/cltl-knowledgerepresentation/blob/main/src/cltl/brain/ontologies/integration.ttl)

This T-box combines various ontologies for repersenting the world, events and interactions, among others.

Given the T-box, interaction data can be pushed to the repository with details on the events that the interlocutors talk about.

## Populating the A-Box with interactions and event details

### Defining the context for the interaction

Every interaction consists of one or more interaction events that are part of a context. Before we can push interaction data to the Knowledge Graph, we first need to define an instance of a context of which the interaction is a subevent.

For this we use the ```capsule_context``` function which we give a dictionary (capsule) with the basic properties of the context and a unique identifier.

In [4]:
# Define contextual features
context_id = getrandbits(8)
place_id = getrandbits(8)
location = requests.get("https://ipinfo.io").json()
start_date = date(2021, 3, 12)

scenario_context = {"context_id": context_id,
                            "date": start_date,
                            "place": "Piek's office",
                            "place_id": place_id,
                            "country": location['country'],
                            "region": location['region'],
                            "city": location['city']}
brain.capsule_context(scenario_context)

2026-06-02 17:21:39 -     INFO -                                    cltl.brain.LongTermMemory - Context: context171


{'response': '204',
 'context': {'context_id': 171,
  'date': datetime.date(2021, 3, 12),
  'place': "Piek's office",
  'place_id': 133,
  'country': 'NL',
  'region': 'North Holland',
  'city': 'Amsterdam'},
 'rdf_log_path': PosixPath('graph/2026-06-02-17-08/brain_log_2026-06-02-17-21-39-720249')}

### Defining event-centric capsules for the turns in a conversation

Given the context, we can now create  a list of interaction events that we push to the repository to be added as subevents of the context. Each element in the list ```conversation_capsules``` that is shown below has a number of obligatory elements:

* [chat]: a unique identifier for the chat interaction
* [turn]: a unique identifier for the turn in the chat
* [author]: a triple dictionary that identifies the author
* [utterance]: the utterance itself
* [utterance_type]: the kind of utterance
* [position]: the offset position that marks the expression(s) in the utterance that mention the information 
* [event_details]: the information as a list of properties of events, where the subject identifies the event
* [perspective]: the perspective of the author (the source) on the event details as a claim
* [timestamp]: when the utterance took place
* [context_id]: the context identifier that was pushed previously.

In [13]:
conversation_capsules = [{"chat": 12,
 "turn": 1,
 "author": {"label": "piek", "type": ["person"], 'uri': "http://cltl.nl/leolani/friends/piek-1"},
 "utterance": "John drinks beer",
 "utterance_type": UtteranceType.STATEMENT,
 "position": "0-15",
 "event_details": [
     {
     "subject": {"label": "drink", "type": ["event"],
                 "uri": "http://cltl.nl/leolani/world/drink-2"
                 },
     "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
     "object": {"label": "john", "type": ["person"],
                "uri": "http://cltl.nl/leolani/world/john-1"
                }

     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
         "object": {"label": "beer", "type": ["drink"],
                    "uri": "http://cltl.nl/leolani/world/beer"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasPlace", "uri": "sem:hasPlace"},
         "object": {"label": "pub", "type": ["place"],
                    "uri": "http://cltl.nl/leolani/world/pub"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasTime", "uri": "sem:hasTime"},
         "object": {"label": "8-3-2026", "type": ["date"],
                    "uri": ""}
     }
 ],
 "perspective": {
     "certainty": 1,
     "polarity": 1,
     "sentiment": 1
 },
 "timestamp": datetime.combine(start_date, datetime.now().time()),
 "context_id": context_id
 },
            {"chat": 12,
 "turn": 2,
 "author": {"label": "luis", "type": ["person"], 'uri': "http://cltl.nl/leolani/friends/luis-1"},
 "utterance": "John does not drink beer",
 "utterance_type": UtteranceType.STATEMENT,
 "position": "0-15",
 "event_details": [
     {
     "subject": {"label": "drink", "type": ["event"],
                 "uri": "http://cltl.nl/leolani/world/drink-2"
                 },
     "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
     "object": {"label": "john", "type": ["person"],
                "uri": "http://cltl.nl/leolani/world/john-1"
                }

     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
         "object": {"label": "beer", "type": ["drink"],
                    "uri": "http://cltl.nl/leolani/world/beer"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasPlace", "uri": "sem:hasPlace"},
         "object": {"label": "pub", "type": ["place"],
                    "uri": "http://cltl.nl/leolani/world/pub"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasTime", "uri": "sem:hasTime"},
         "object": {"label": "8-3-2026", "type": ["date"],
                    "uri": ""}
     }
 ],
 "perspective": {
     "certainty": 1,
     "polarity": -1,
     "sentiment": -1
 },
 "timestamp": datetime.combine(start_date, datetime.now().time()),
 "context_id": context_id
 }]

The event details create an event-centric Knowledge graph. This means that the subject defines an instance of an event through a unique URI, while the predicate and object express values for properties of the event. We follow here the Simple Event Ontology (SEM, van Hage et al 2011) that is also part of the T-box. Typically, the predicates are semantic roles that define the event. In SEM, these are hasActor, hasPlace and hasTime but also more specific semantic roles can be defined. It is advised to define these specific roles as subclasses of the SEM predicates.

### Pushing the turn capsules to the Graph

The above conversational data is pushed to the repository. It will add the interaction events as such, i.e. chats, turns, but also the events reported. All as part of the context.

In [14]:
for capsule in conversation_capsules:
    brain.capsule_event(capsule, reason_types=True, return_thoughts=False, create_label=True)

2026-06-02 15:26:47 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasActor_john [event_->_person])
2026-06-02 15:26:47 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasActor_beer [event_->_drink])
2026-06-02 15:26:47 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasPlace_pub [event_->_place])
2026-06-02 15:26:47 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasTime_8-3-2026 [event_->_date])
2026-06-02 15:26:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasActor_john [event_->_person])
2026-06-02 15:26:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: drink_hasActor_beer [event_->_drink])
2026-06-02 15:26:48 -     INFO -                                   

You can now inspect the event_sandbox in GraphDB to find the event data, which should look as follows.

In [16]:
# import image module
from IPython.display import Image

Image(url="ekg-notebook.png", width=800, height=600)